# K13 · Evaluate all samples (step 6, CPU — uses no GPU quota)

Computes the metric tables (`scripts/evaluate.py`: SIFID, diversity, NIQE/MUSIQ/NIMA, patch statistics, copying,
structure) for every experiment it finds, several jobs in parallel. Results: `eval/<experiment>/<job>/per_image.csv`
and `per_sample.csv`, all in `cs787_k13_eval.zip`.

**Inputs to attach first** (*Add Input*):
- *Your Work* → the latest versions of **K10**, **K11**, **K12** (their `cs787_*.zip` are found automatically);
- the Colab results of step 2: download the folder `MyDrive/CS787/outputs/phase4_reproduction` from Google Drive
  (right-click → Download, gives a zip), then *Add Input* → *Upload* → *New Dataset* → drop the zip → Create.
Already-evaluated jobs are skipped, so you can rerun it after attaching more results.

**How to run (Kaggle):**
1. Right panel → *Session options*: **Accelerator = None (CPU)**, **Internet = On**.
2. Top right **Save Version** → *Save & Run All (Commit)* → Save. It runs in the background (you can close the tab);
   watch it under *View Active Events* / the version's **Logs** (progress lines every 2 min).
3. When it finishes, the result is the version's **Output**: `cs787_k13_eval.zip`.
4. **Not finished?** (it stops itself before 2 hours) → *Add Input* → *Your Work* → this notebook's
   latest version → then *Save Version* again. It continues where it stopped.

In [1]:
import time
T_START = time.time()                      # session clock: work stops before SESSION_HOURS
NOTEBOOK = "K13_evaluate"
GITHUB_REPO   = "https://github.com/rishabh-vr/CS787-Single-Image-Diffusion-Editor.git"
GITHUB_BRANCH = "main"
SID_COMMIT    = "85381e3c6f29fc07144da854d86be07150404716"     # official code, audited commit
SINDDM_COMMIT = "e7107d9fd198928f26e278f99ec428e261726a45"  # SinDDM, audited commit
SESSION_HOURS = 1.9                        # hard stop: everything (incl. zipping) ends before 2 h

GROUPS      = "paper,patch,copy,structure"   # "patch,copy,structure" = fast pass without pretrained networks
MAX_SAMPLES = 20                             # per image (keeps the run < 2 h)
SLOTS       = 2                              # jobs evaluated at the same time (Kaggle CPU sessions have 4 cores)
# experiment folder -> manifests that contain its input images
EVAL = {
    "phase4_reproduction": ["uncond_sinddm15"],
    "phase7_ood":          ["ood_setA", "synthetic_setB"],
    "phase8_ours":         ["sinddm_ood3", "lora8"],
    "phase8_sinddm":       ["uncond_sinddm15", "sinddm_ood3"],
    "phase8_lora":         ["lora8"],
    "phase9_ablations":    ["ablation_subset"],
    "phase9_ab6_ann":      ["ablation_subset"],
}

In [2]:
import os, sys, subprocess, shutil, glob, json
if not os.path.isdir("/kaggle/working"):
    raise SystemExit("This notebook is written for Kaggle (kaggle.com/code).")
WORK, TMP = "/kaggle/working", "/tmp/cs787"
PROJECT = f"{TMP}/project"
shutil.rmtree(PROJECT, ignore_errors=True)
os.makedirs(TMP, exist_ok=True)
r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", GITHUB_BRANCH, GITHUB_REPO, PROJECT],
                   capture_output=True, text=True)
if r.returncode:
    raise SystemExit("Could not download the code from GitHub. Turn Internet ON: right panel -> Session options -> "
                     "Internet (needs a phone-verified Kaggle account), then run again.\n" + r.stderr)
sys.path.insert(0, f"{PROJECT}/src")
os.environ["PYTHONPATH"] = f"{PROJECT}/src" + os.pathsep + os.environ.get("PYTHONPATH", "")
from sidx.kaggle import gather_inputs, unpack_own, pack
from sidx.progress import Bars
print("code:", subprocess.run(["git", "-C", PROJECT, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)
gather_inputs()          # results of earlier versions attached under /kaggle/input (resume)

code: 02e9541 Move Kaggle-synced K12 run into notebooks/kaggle/runs/

[resume] /kaggle/input/notebooks/rishabh0verma/cs787-k12/cs787_k12.zip -> /kaggle/working
[resume] /kaggle/input/notebooks/rishabh0verma/notebook1e5860a744/cs787_k11.zip -> /kaggle/working
[resume] /kaggle/input/notebooks/rishabh0verma/notebook2647ebba42/cs787_k10.zip -> /kaggle/working
[resume] /kaggle/input/datasets/rishabh0verma/cs787-phase4/phase4_reproduction -> /kaggle/working/outputs/phase4_reproduction (366 files)


['/kaggle/input/notebooks/rishabh0verma/cs787-k12/cs787_k12.zip -> /kaggle/working',
 '/kaggle/input/notebooks/rishabh0verma/notebook1e5860a744/cs787_k11.zip -> /kaggle/working',
 '/kaggle/input/notebooks/rishabh0verma/notebook2647ebba42/cs787_k10.zip -> /kaggle/working',
 '/kaggle/input/datasets/rishabh0verma/cs787-phase4/phase4_reproduction -> /kaggle/working/outputs/phase4_reproduction (366 files)']

In [3]:
!pip install -q lpips pytorch-fid pyiqa opencv-python-headless scikit-image scipy pandas
!pip install -q --no-deps -e {PROJECT}
SID_ROOT = f"{TMP}/efficient-SID-code"
if not os.path.exists(SID_ROOT):
    subprocess.run(["git", "clone", "-q", "https://github.com/haojunqiu/efficient-SID-code", SID_ROOT], check=True)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sidx (pyproject.toml) ... done


In [4]:
# natural images (balloons, forest, ...) are third-party and not in the repo: fetch + check sha256 against the manifests
r = subprocess.run([sys.executable, "scripts/prepare_natural.py", "--verify", "--sets", "uncond/sinddm15", "--sid-root", SID_ROOT],
                   cwd=PROJECT, capture_output=True, text=True)
print(r.stdout[-1200:], r.stderr[-1200:])
assert r.returncode == 0, "data check failed (see above)"

uncond/sinddm15: 15 images verified against data/manifests/uncond_sinddm15.csv
 


## Evaluate (parallel; one bar per experiment, counted in images)

In [5]:
from sidx.gpupool import Task, run_pool
unpack_own()
OUT_ROOT, EVAL_ROOT = f"{WORK}/outputs", f"{WORK}/eval"
threads = max(1, (os.cpu_count() or 2) // SLOTS)
tasks, plan = [], []
for exp, mans in EVAL.items():
    for jj in sorted(glob.glob(f"{OUT_ROOT}/{exp}/*/job.json")):
        job_dir = os.path.dirname(jj); job = os.path.basename(job_dir)
        n_img = sum(1 for d in glob.glob(f"{job_dir}/*/") if glob.glob(f"{d}seed_*.png"))
        plan.append((exp, job, n_img))
        cmd = [sys.executable, "scripts/evaluate.py", f"{OUT_ROOT}/{exp}", "--only-job", job, "--groups", GROUPS,
               "--max-samples", str(MAX_SAMPLES), "--eval-root", EVAL_ROOT, "--no-progress"]
        for m in mans:
            cmd += ["--manifest", f"{PROJECT}/data/manifests/{m}.csv"]
        tasks.append(Task(f"{exp}:{job}", cmd, cwd=PROJECT, env={"OMP_NUM_THREADS": str(threads), "MKL_NUM_THREADS": str(threads)}))
print({e: sum(1 for p in plan if p[0] == e) for e in EVAL}, "jobs found")

def poll():
    res = {}
    for exp, job, n_img in plan:
        d = f"{EVAL_ROOT}/{exp}/{job}"
        try:
            done = json.load(open(f"{d}/progress.json"))["done"]
        except Exception:
            done = n_img if os.path.exists(f"{d}/stamp.json") else 0
        a, b = res.get(exp, (0, 0))
        res[exp] = (a + min(done, n_img), b + n_img)
    return res

try:
    run_pool(tasks, poll, slots=SLOTS, n_gpus=0, log_dir=f"{WORK}/logs/{NOTEBOOK}", unit="image",
             deadline=T_START + SESSION_HOURS * 3600 - 10 * 60)
finally:
    pack("k13_eval", kinds=("eval",), remove=False)

{'phase4_reproduction': 4, 'phase7_ood': 3, 'phase8_ours': 2, 'phase8_sinddm': 3, 'phase8_lora': 1, 'phase9_ablations': 13, 'phase9_ab6_ann': 0} jobs found
[pool] 26 tasks on 2 slot(s), 0 GPU(s); logs in /kaggle/working/logs/K13_evaluate
[11:01:45] phase4_reproduction: 0/7 images (0%) | 0.0/min | ETA ?
[11:01:45] phase7_ood: 0/48 images (0%) | 0.0/min | ETA ?
[11:01:45] phase8_ours: 0/8 images (0%) | 0.0/min | ETA ?
[11:01:45] phase8_sinddm: 0/9 images (0%) | 0.0/min | ETA ?
[11:01:45] phase8_lora: 0/8 images (0%) | 0.0/min | ETA ?
[11:01:45] phase9_ablations: 0/50 images (0%) | 0.0/min | ETA ?
[11:01:45] tasks 0/26 finished | running: worker0: phase4_reproduction:core_paper_fp32__3e7b0bea (0 min), worker1: phase4_reproduction:official_T40_eta1_fp32__8fd9ff38 (0 min)
[11:03:45] phase4_reproduction: 0/7 images (0%) | 0.0/min | ETA ?
[11:03:45] phase7_ood: 1/48 images (2%) | 0.5/min | ETA 1h34m
[11:03:45] phase8_ours: 0/8 images (0%) | 0.0/min | ETA ?
[11:03:45] phase8_sinddm: 0/9 images

## Quick look: per-job means of the main metrics

In [6]:
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200)
rows = []
for f in sorted(glob.glob(f"{EVAL_ROOT}/*/*/per_image.csv")):
    df = pd.read_csv(f)
    exp, job = f.split("/")[-3], f.split("/")[-2]
    rows.append({"experiment": exp, "job": job.split("__")[0], "images": len(df),
                 **df.select_dtypes("number").mean().to_dict()})
summary = pd.DataFrame(rows)
summary.to_csv(f"{WORK}/eval_summary.csv", index=False)
summary.round(4)

,experiment,job,images,n_samples,height,width,pixel_div,pixel_div_raw,lpips_div,near_dup_r50,...,junctions_per_kpx,seg_len_w1_rel,seg_angle_w1_deg,mean_seg_len_ratio,grey_ratio,edge_density_ratio,grad_mag_ratio,periodicity_ratio,spectral_peak_recall,orientation_conformity
0,phase4_reproduction,official_paper_fp32,7,20.0,180.8571,267.0000,0.6752,0.1494,0.5006,0.2000,...,45.7221,0.0941,5.9458,1.0323,0.9804,0.9533,0.9393,2.7485,0.9393,NaN
1,phase7_ood,setA_bleed_3scales_fp32,1,6.0,110.0000,256.0000,0.6888,0.1691,0.4212,1.0000,...,20.7250,0.1478,8.6283,0.9153,1.1564,1.0973,1.0969,1.0901,0.5625,NaN
2,phase7_ood,setA_paper_fp32,23,6.0,241.4783,248.7391,0.6646,0.1757,0.5104,0.5435,...,32.2746,0.1347,4.9307,0.9255,1.0167,0.9991,0.9886,0.7800,0.7563,NaN
3,phase7_ood,setB_paper_fp32,24,6.0,128.0000,128.0000,0.6159,0.2131,0.1120,0.7708,...,24.0665,0.0555,3.5027,0.9968,1.0627,1.0363,1.0390,0.8756,0.9592,0.8497
4,phase8_lora,lora_r8_s400,8,15.0,222.6250,266.0000,0.8357,0.2187,0.4275,0.2667,...,35.3613,0.1508,5.4637,1.0157,1.3945,1.1690,1.2724,1.0702,0.8302,NaN
5,phase8_ours,ours_lora8_ood5,5,10.0,255.6000,247.6000,0.7783,0.2303,0.5305,0.6000,...,29.0573,0.1632,2.9149,0.8503,1.0876,1.0354,1.0275,0.6317,0.8125,NaN
6,phase8_ours,ours_sinddm_ood3_128px,3,10.0,128.0000,126.6667,0.5333,0.1607,0.1289,0.7667,...,35.7928,0.0412,2.4446,0.9961,1.0381,1.0270,1.0259,0.8656,0.9667,NaN
7,phase8_sinddm,sinddm_pretrained_full,3,20.0,167.6667,296.6667,0.4318,0.0954,0.3800,0.7167,...,40.4421,0.1126,5.1395,0.9658,0.9731,0.9532,0.8421,1.3372,0.9792,NaN
8,phase8_sinddm,sinddm_steps1500,3,20.0,128.0000,126.6667,0.4679,0.1401,0.3803,0.6667,...,22.6506,0.5646,12.8010,1.1665,1.0611,0.6810,0.4785,0.3004,0.5187,NaN
9,phase8_sinddm,sinddm_steps3000,3,20.0,128.0000,126.6667,0.4409,0.1363,0.3992,0.6667,...,19.5059,0.3994,10.6028,1.0539,0.9150,0.6141,0.4428,0.3343,0.5896,NaN


In [7]:
os.makedirs(f"{WORK}/logs", exist_ok=True)
!pip freeze > {WORK}/logs/{NOTEBOOK}_pip_freeze.txt
print("finished; output files:", sorted(os.listdir(WORK)))

finished; output files: ['__notebook__.ipynb', 'checkpoints', 'cs787_k13_eval.zip', 'eval', 'eval_summary.csv', 'logs', 'outputs', 'tables']
